# 🚀 FaceFusion Backend for PromptHub

This notebook sets up a **Flask API server** with a `/swap` endpoint that the PromptHub Face Swap page calls.

### Steps:
1. Run **Cell 1** – Install dependencies
2. Run **Cell 2** – Clone & set up FaceFusion
3. Run **Cell 3** – Start the Flask server + ngrok tunnel
4. Copy the **ngrok URL** printed at the end and paste it into PromptHub

> ⚠️ Enable **GPU** in Kaggle: Settings → Accelerator → GPU T4 x2
>
> ⚠️ After Cell 1 completes, **restart the runtime** (Runtime → Restart runtime), then run Cell 2 and Cell 3.

In [ ]:
# Cell 1: Install dependencies
# Pin numpy to 1.x FIRST — insightface/onnxruntime are incompatible with numpy 2.x
# (fixes: ImportError: cannot import name '_center' from 'numpy._core.umath')
!pip install -q 'numpy==1.26.4'
!pip install -q flask flask-cors pyngrok onnxruntime-gpu insightface opencv-python-headless
!pip install -q gfpgan basicsr facexlib realesrgan

import numpy as np
print(f'✅ numpy {np.__version__} ready')
print('✅ All dependencies installed')
print('⚠️  Now RESTART the runtime, then run Cell 2 and Cell 3.')

In [ ]:
# Cell 2: Clone FaceFusion and download models
import os

if not os.path.exists('/kaggle/working/facefusion'):
    !git clone https://github.com/facefusion/facefusion.git /kaggle/working/facefusion
    %cd /kaggle/working/facefusion
    !pip install -q -r requirements.txt
else:
    %cd /kaggle/working/facefusion
    print('✅ FaceFusion already cloned')

# Download required models
os.makedirs('/kaggle/working/facefusion/.assets/models', exist_ok=True)

models = {
    'inswapper_128.onnx': 'https://huggingface.co/ezioruan/inswapper_128.onnx/resolve/main/inswapper_128.onnx',
    'GFPGANv1.4.pth': 'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth',
}

for model_name, url in models.items():
    model_path = f'/kaggle/working/facefusion/.assets/models/{model_name}'
    if not os.path.exists(model_path):
        print(f'⬇️  Downloading {model_name}...')
        !wget -q -O "{model_path}" "{url}"
        print(f'✅ {model_name} downloaded')
    else:
        print(f'✅ {model_name} already exists')

print('\n✅ All models ready!')

In [ ]:
# Cell 3: Start Flask server with /swap endpoint + ngrok tunnel
import os
import sys
import uuid
import threading
import subprocess
import cv2
import numpy as np
from flask import Flask, request, jsonify, send_file
from flask_cors import CORS
from pyngrok import ngrok
import insightface
from insightface.app import FaceAnalysis
from gfpgan import GFPGANer
import io

# ── Setup paths ────────────────────────────────────────────────────────────────
WORK_DIR = '/kaggle/working'
MODEL_DIR = f'{WORK_DIR}/facefusion/.assets/models'
UPLOAD_DIR = f'{WORK_DIR}/uploads'
os.makedirs(UPLOAD_DIR, exist_ok=True)

# ── Load InsightFace (face swapper) ───────────────────────────────────────────
print('🔄 Loading InsightFace models...')
face_app = FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
face_app.prepare(ctx_id=0, det_size=(640, 640))

swapper = insightface.model_zoo.get_model(
    f'{MODEL_DIR}/inswapper_128.onnx',
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
)
print('✅ InsightFace loaded')

# ── Load GFPGAN (face enhancer) ───────────────────────────────────────────────
print('🔄 Loading GFPGAN enhancer...')
enhancer = GFPGANer(
    model_path=f'{MODEL_DIR}/GFPGANv1.4.pth',
    upscale=1,
    arch='clean',
    channel_multiplier=2,
    bg_upsampler=None
)
print('✅ GFPGAN loaded')

# ── Flask App ─────────────────────────────────────────────────────────────────
app = Flask(__name__)
CORS(app)  # Allow cross-origin requests from PromptHub

@app.route('/', methods=['GET'])
def health():
    return jsonify({'status': 'running', 'gpu': 'enabled'})

@app.route('/swap', methods=['POST'])
def swap_faces():
    """POST /swap
    Form fields:
        source_image: the face you want to transplant (file)
        target_image: the image where the face goes (file)
    Returns:
        JPEG image of the swapped result
    """
    try:
        if 'source_image' not in request.files or 'target_image' not in request.files:
            return jsonify({'error': 'Missing source_image or target_image'}), 400

        source_file = request.files['source_image']
        target_file = request.files['target_image']

        # Read images
        source_bytes = np.frombuffer(source_file.read(), np.uint8)
        target_bytes = np.frombuffer(target_file.read(), np.uint8)
        source_img = cv2.imdecode(source_bytes, cv2.IMREAD_COLOR)
        target_img = cv2.imdecode(target_bytes, cv2.IMREAD_COLOR)

        if source_img is None or target_img is None:
            return jsonify({'error': 'Could not decode one or both images'}), 400

        # Detect faces
        source_faces = face_app.get(source_img)
        target_faces = face_app.get(target_img)

        if not source_faces:
            return jsonify({'error': 'No face detected in source image'}), 400
        if not target_faces:
            return jsonify({'error': 'No face detected in target image'}), 400

        # Swap face (use the largest/most prominent face from source)
        source_face = sorted(source_faces, key=lambda f: f.bbox[2] - f.bbox[0], reverse=True)[0]
        result_img = target_img.copy()

        for target_face in target_faces:
            result_img = swapper.get(result_img, target_face, source_face, paste_back=True)

        # Enhance with GFPGAN
        _, _, result_img = enhancer.enhance(
            result_img,
            has_aligned=False,
            only_center_face=False,
            paste_back=True
        )

        # Encode result as JPEG and return
        _, buffer = cv2.imencode('.jpg', result_img, [cv2.IMWRITE_JPEG_QUALITY, 95])
        return send_file(
            io.BytesIO(buffer.tobytes()),
            mimetype='image/jpeg',
            as_attachment=False
        )

    except Exception as e:
        import traceback
        traceback.print_exc()
        return jsonify({'error': str(e)}), 500

# ── Start ngrok tunnel ────────────────────────────────────────────────────────
# Optional: set your ngrok auth token for a stable URL
# ngrok.set_auth_token('YOUR_NGROK_AUTH_TOKEN')

public_url = ngrok.connect(5000)
print('\n' + '='*60)
print(f'🌐 PromptHub Server URL: {public_url}')
print(f'📋 Paste this URL into PromptHub Face Swap page')
print('='*60 + '\n')

# Run Flask (blocking)
app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)